In [1]:
import sys
import os
import numpy as np
import pandas as pd
from category_encoders import TargetEncoder
from sklearn.svm import SVC
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

# ==========================================
# 1. CONFIGURACIÓN Y CARGA DE DATOS
# ==========================================
ruta_raiz = r"C:\Users\marli\Downloads\vehicle-emissions-classification-main (1)\vehicle-emissions-classification-main"
if ruta_raiz not in sys.path:
    sys.path.append(ruta_raiz)

from src.preprocessing import LowDoc, franchise_code, urban_rural, RevLineCr, newExists, base_cleaning

print("Iniciando tubería de preprocesamiento con Bagging sobre Máquinas de Vectores de Soporte...")

ruta_train = os.path.join(ruta_raiz, "data", "train.csv")
ruta_test = os.path.join(ruta_raiz, "data", "test_nolabel.csv")

df_train = pd.read_csv(ruta_train)
df_test = pd.read_csv(ruta_test)
ids_test = df_test['id'].copy()

# ==========================================
# 2. TUBERÍA DE PREPROCESAMIENTO BASE
# ==========================================
def preprocess_base_features(df: pd.DataFrame, local_state: str = "IL") -> pd.DataFrame:
    df_out = df.copy()
    df_out = base_cleaning.clean_base_columns(df_out, local_state=local_state)

    # Recuperación de variables monetarias y laborales en formato numérico
    if 'DisbursementGross' in df_out.columns:
        df_out['Disbursement_Raw'] = pd.to_numeric(
            df_out['DisbursementGross'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False), 
            errors='coerce'
        ).fillna(0)
        
    for col in ['CreateJob', 'RetainedJob', 'NoEmp']:
        if col in df_out.columns:
            df_out[col + '_Raw'] = pd.to_numeric(df_out[col], errors='coerce').fillna(0)

    # Construcción de variable temporal: antigüedad del préstamo en días
    if "ApprovalDate" in df_out.columns:
        fechas = pd.to_datetime(df_out["ApprovalDate"], format='%d-%b-%y', errors='coerce')
        df_out["LoanAgeDays"] = (pd.Timestamp("2020-01-01") - fechas).dt.days.fillna(0)

    # Aplicación de módulos de preprocesamiento específicos por variable
    df_out = LowDoc.preprocess_lowdoc(df_out, option="C")
    df_out = RevLineCr.preprocess_revlinecr(df_out, option="C")
    df_out = franchise_code.preprocess_franchise_code(df_out, option="binary")
    df_out = urban_rural.preprocess_urban_rural(df_out, option="onehot")
    df_out = newExists.preprocess_newexist(df_out, option="A")

    # Eliminación de columnas no informativas o redundantes
    cols_to_drop = [
        "Zone_Undefined", "lowdoc_is_missing", "newexist_missing_or_invalid", "lowdoc_is_nonstandard",
        "State", "Zip", "BankState", "ApprovalDate", "ApprovalFY", 
        "DisbursementDate", "Name", "LoanNr_ChkDgt", "BalanceGross", "DisbursementGross",
        "CreateJob", "RetainedJob", "NoEmp", "id"
    ]
    return df_out.drop(columns=[c for c in cols_to_drop if c in df_out.columns], errors='ignore')

df_train_clean = preprocess_base_features(df_train)
df_train_clean['Accept'] = df_train_clean['Accept'].astype(float)
df_train_clean = df_train_clean.dropna(subset=['Accept'])

df_test_clean = preprocess_base_features(df_test)

y_completo = df_train_clean["Accept"].values
X_completo = df_train_clean.drop(columns=["Accept"])

X_completo, X_test_final = X_completo.align(df_test_clean, join='left', axis=1, fill_value=0)

# ==========================================
# 3. VALIDACIÓN CRUZADA ESTRATIFICADA Y CODIFICACIÓN POR OBJETIVO
# ==========================================
print("Iniciando validación cruzada estratificada con prevención de fuga de datos...")

# Definición del clasificador base: Máquina de Vectores de Soporte con kernel RBF
# El parámetro probability=True es obligatorio para obtener estimaciones de probabilidad
svc_base = SVC(
    kernel='rbf', 
    probability=True,
    class_weight='balanced', 
    random_state=42
)

# Ensamble de Bagging sobre el clasificador base SVC
# Se utilizan 50 estimadores con submuestreo del 80% por iteración
bagging_svc = BaggingClassifier(
    estimator=svc_base,
    n_estimators=50,
    max_samples=0.8,
    n_jobs=-1,
    random_state=42
)

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_probs = np.zeros(len(X_completo))
probs_test_acumuladas = np.zeros(len(X_test_final))

fold = 1
for tr_idx, va_idx in kf.split(X_completo, y_completo):
    print(f" -> Procesando partición {fold} de 5...")
    
    X_tr, y_tr = X_completo.iloc[tr_idx].copy(), y_completo[tr_idx]
    X_va, y_va = X_completo.iloc[va_idx].copy(), y_completo[va_idx]
    X_te = X_test_final.copy()
    
    # Codificación por objetivo aplicada exclusivamente sobre la partición de entrenamiento
    # para evitar fuga de información hacia la partición de validación
    cat_cols = ["City", "Bank"]
    cols_to_encode = [c for c in cat_cols if c in X_tr.columns]
    
    if cols_to_encode:
        te = TargetEncoder(cols=cols_to_encode, smoothing=0.3)
        X_tr[cols_to_encode] = te.fit_transform(X_tr[cols_to_encode], y_tr)
        X_va[cols_to_encode] = te.transform(X_va[cols_to_encode])
        X_te[cols_to_encode] = te.transform(X_te[cols_to_encode])
    
    # Estandarización obligatoria: los modelos SVC son sensibles a la escala de las características
    X_tr = X_tr.select_dtypes(include=['number', 'bool']).fillna(0).astype(float)
    X_va = X_va.select_dtypes(include=['number', 'bool']).fillna(0).astype(float)
    X_te = X_te.select_dtypes(include=['number', 'bool']).fillna(0).astype(float)
    
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_va_scaled = scaler.transform(X_va)
    X_te_scaled = scaler.transform(X_te)
    
    # Entrenamiento del ensamble y acumulación de probabilidades
    bagging_svc.fit(X_tr_scaled, y_tr)
    oof_probs[va_idx] = bagging_svc.predict_proba(X_va_scaled)[:, 1]
    probs_test_acumuladas += bagging_svc.predict_proba(X_te_scaled)[:, 1]
    
    fold += 1

# ==========================================
# 4. OPTIMIZACIÓN DEL UMBRAL DE DECISIÓN Y PREDICCIÓN FINAL
# ==========================================
print("\nOptimizando umbral de decisión mediante maximización del Macro F1-score...")
mejor_umbral = 0.50
mejor_f1 = 0

for u in np.arange(0.35, 0.65, 0.01):
    f = f1_score(y_completo, (oof_probs >= u).astype(int), average='macro')
    if f > mejor_f1:
        mejor_f1, mejor_umbral = f, u

print(f" -> Umbral óptimo seleccionado: {mejor_umbral:.2f} (Macro F1 fuera de muestra: {mejor_f1:.4f})")

print("\nGenerando archivo de predicciones finales...")
predicciones_test = (probs_test_acumuladas / 5.0 >= mejor_umbral).astype(int)

df_entrega = pd.DataFrame({
    "id": ids_test,
    "Accept": predicciones_test
})

ruta_csv = os.path.join(ruta_raiz, "submission_bagging_svc.csv")
df_entrega.to_csv(ruta_csv, index=False)

print(f"Proceso finalizado. Archivo de entrega generado en: {ruta_csv}")

Iniciando Experimento de Contraste: Bagging con SVC (Estable)...
Iniciando Stratified K-Fold CV (Prevención de Data Leakage)...
 -> Entrenando Fold 1 (Puede tardar por la complejidad del SVC)...
 -> Entrenando Fold 2 (Puede tardar por la complejidad del SVC)...
 -> Entrenando Fold 3 (Puede tardar por la complejidad del SVC)...
 -> Entrenando Fold 4 (Puede tardar por la complejidad del SVC)...
 -> Entrenando Fold 5 (Puede tardar por la complejidad del SVC)...

Optimizando umbral de decisión basado en Macro F1-score...
 -> Umbral óptimo (Bagging SVC): 0.58 (Macro F1 Out-of-Fold: 0.6962)

Generando CSV de predicciones finales...
¡Proceso completado! Archivo generado en: C:\Users\marli\Downloads\vehicle-emissions-classification-main (1)\vehicle-emissions-classification-main\submission_bagging_svc.csv
